In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Solution.LevelSetTools;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
#r "LsTest.dll"
using BoSSS.Application.LsTest;

# Load Sessions

Init Workflowmanagement and Project

In [ ]:
BoSSSshell.WorkflowMgm.Init("SwirlingFlow_TemporalConvergence");
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

Project name is set to 'SwirlingFlow_TemporalConvergence'.
Opening existing database '\\hpccluster\hpccluster-scratch\rieckmann\SwirlingFlow_TemporalConvergence'.


In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);

In [ ]:
sessions

#0: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p3_H4_t4_EvoPrescribedVelocity	06/30/2022 18:18:58	a408615b...
#1: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p3_H4_t3_EvoPrescribedVelocity	06/30/2022 18:18:14	8ca877e9...
#2: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p3_H4_t2_EvoPrescribedVelocity	06/30/2022 18:18:00	f8e0ee42...
#3: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p3_H4_t1_EvoPrescribedVelocity	06/30/2022 18:17:18	b0a23d05...
#4: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p2_H4_t4_EvoPrescribedVelocity	06/30/2022 18:17:05	bb5425ac...
#5: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p2_H4_t3_EvoPrescribedVelocity	06/30/2022 18:16:24	e6e43068...
#6: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p2_H4_t2_EvoPrescribedVelocity	06/30/2022 18:16:11	ce34062f...
#7: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p2_H4_t1_EvoPrescribedVelocity	06/30/2022 18:15:38	1a996bad...
#8: SwirlingFlow_TemporalConvergence	SwirlingFlow_T_p3_H4_t4_EvoStokesExtension	

In [ ]:
using System.IO;

In [ ]:
// var s = sessions.Pick(0);
// s.Export().To(Path.GetFullPath("./Plots/"+s.Name)).WithSupersampling(2).Do();

Starting export process... Data will be written to the directory: d:\rieckmann\BoSSS-Repos\BoSSS-Master\public\examples\levelset\Plots\SwirlingFlow_T_p3_H4_t4_EvoPrescribedVelocity


In [ ]:
var tab = sessions.GetSessionTable();

In [ ]:
var Evos = tab.GetColumn<string>("id:Evo").Distinct();
Evos.ForEach(e => Console.WriteLine(e));

var hRes = tab.GetColumn<int>("id:Res").Distinct();
hRes.ForEach(e => Console.WriteLine(e));

var pRes = tab.GetColumn<int>("id:Degree").Distinct();
pRes.ForEach(e => Console.WriteLine(e));

PrescribedVelocity
StokesExtension
FastMarching
Prescribed
4
3
2


In [ ]:
List<Plot2Ddata> plts = new List<Plot2Ddata>();

foreach(string Evo in Evos){
    foreach(int p in pRes){
        foreach(int h in hRes){

            // select and sort
            var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToInt32(s.KeysAndQueries["id:Res"]) == h && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
            Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:dt"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:dt"])));
            sess.ForEach(s => Console.WriteLine(s.Name));

            var Ref = new BoSSS.Application.LsTest.LevelSetUnitTests.LevelSetSwirlingFlowTest(2, p, false, 0.0); // to get the "Phi" Functions etc.

            Func<ITimestepInfo, double> errorFunctional = ts => {
                var PhiDG = ts.Fields.Single(f => f.Identification == "PhiDG");
                var PhiCG = ts.Fields.Single(f => f.Identification == "Phi");

                // use the same domain for all error evaluations
                var RefLS = new LevelSet(PhiCG.Basis, "PhiRef");
                RefLS.ProjectField(1.0, Ref.GetPhi()[0].Vectorize().SetTime(0.0));
                LevelSetTracker LsTrk = new LevelSetTracker((GridData)PhiCG.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] { "A", "B" }, RefLS);
                LsTrk.UpdateTracker(0.0);

                double err = 0.0;
                err = PhiDG.L2Error(Ref.GetPhi()[0].Vectorize().SetTime(0.0), 16, new BoSSS.Foundation.Quadrature.CellQuadratureScheme(true, LsTrk.Regions.GetNearFieldMask(1)));
                return err;
            };

            var errs = sess.Select(s => errorFunctional(s.Timesteps.Last())).ToArray();
            errs.ForEach(err => Console.WriteLine(err));

            var plt = sess.ToTimeConvergenceData(errorFunctional);
            plts.Add(plt);

            plt.Title = "Res: " + h + " Degree: " + p + " Evo: " + Evo;
        }
    }
}

SwirlingFlow_T_p3_H4_t4_EvoPrescribedVelocity
SwirlingFlow_T_p3_H4_t3_EvoPrescribedVelocity
SwirlingFlow_T_p3_H4_t2_EvoPrescribedVelocity
SwirlingFlow_T_p3_H4_t1_EvoPrescribedVelocity
3.611822885752114E-05
4.647679514109704E-05
6.787156281143865E-05
0.00013352362887189697
SwirlingFlow_T_p2_H4_t4_EvoPrescribedVelocity
SwirlingFlow_T_p2_H4_t3_EvoPrescribedVelocity
SwirlingFlow_T_p2_H4_t2_EvoPrescribedVelocity
SwirlingFlow_T_p2_H4_t1_EvoPrescribedVelocity
7.470757094554422E-05
9.961048525621499E-05
0.00014941630911440633
0.00029883065781738247
SwirlingFlow_T_p3_H4_t4_EvoStokesExtension
SwirlingFlow_T_p3_H4_t3_EvoStokesExtension
SwirlingFlow_T_p3_H4_t2_EvoStokesExtension
SwirlingFlow_T_p3_H4_t1_EvoStokesExtension
3.3590191546408436E-05
4.4671566890613587E-05
6.689392963417587E-05
0.00013368406225353584
SwirlingFlow_T_p2_H4_t4_EvoStokesExtension
SwirlingFlow_T_p2_H4_t3_EvoStokesExtension
SwirlingFlow_T_p2_H4_t2_EvoStokesExtension
SwirlingFlow_T_p2_H4_t1_EvoStokesExtension
7.508250726529116E

In [ ]:
foreach(var plt in plts){
    Console.WriteLine("Slope: " + plt.Regression().First().Value);
    display(plt.PlotNow());
}

Slope: 0.9463822245837801
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -5 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Res: 4 Degree: 3 Evo: PrescribedVelocity 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354

Slope: 1.0000005475719451
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -5 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Res: 4 Degree: 2 Evo: PrescribedVelocity 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354

Slope: 0.9966237668059464
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


...

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 $10 -1 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Res: 4 Degree: 2 Evo: FastMarching 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354

Slope: 0
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Gnuplot Error: Warning: empty y range [1.58333e-009:1.58333e-009], adjusting to [1.56749e-009:1.59916e-009]



<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -9 $ 
 
 
 
 
 $10 -8 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Res: 4 Degree: 3 Evo: Prescribed 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354

Slope: 0
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Gnuplot Error: Warning: empty y range [7.88103e-008:7.88103e-008], adjusting to [7.80222e-008:7.95984e-008]



<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -8 $ 
 
 
 
 
 $10 -7 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Res: 4 Degree: 2 Evo: Prescribed 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354 
 
 
 7ec63e62-e507-4b75-945b-15dc00e42354